In [ ]:
# 04_Voting

In [1]:
import sys
import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from scipy.stats import mode
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# -------------------------- Global Configuration --------------------------
plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "axes.unicode_minus": False,
})

MODELS = [
    'DenseNet201', 'NASNetLarge', 'EfficientNetB7', 'Xception',
    'InceptionResNetV2', 'VGG19', 'ResNet50V2', 'MobileNetV3Large', 'EfficientNetV2S'
]

BASE_DIR = r'C:\Users\zhd\LungCancer'
OUTPUT_DIR = os.path.join(BASE_DIR, '03.Ensemble learning')

# -------------------------- Utility Functions --------------------------
def load_all_predictions():
    """ Load prediction csv from each base model
    Return dict containing filename, label, dataset flag and model outputs """
    dfs = {}
    for m in MODELS:
        path = os.path.join(BASE_DIR, m, 'all_predictions.csv')
        df = pd.read_csv(path)
        dfs[m] = df

    ref = dfs[MODELS[0]]
    filenames = ref['Filename'].values
    y_true = ref['True_Label'].values.astype(int)
    datasets = ref['Dataset'].values

    X_raw = np.column_stack([dfs[m]['Raw_Score'].values for m in MODELS])
    X_prob = np.column_stack([dfs[m]['Probability'].values for m in MODELS])
    X_pred = np.column_stack([dfs[m]['Predicted_Label'].values.astype(int) for m in MODELS])

    return {
        'filenames': filenames,
        'y_true': y_true,
        'datasets': datasets,
        'X_raw': X_raw,
        'X_prob': X_prob,
        'X_pred': X_pred,
        'model_names': MODELS
    }


def split_by_dataset(data, X=None):
    """ Split feature matrix and label by Dataset flag: Train / Validation / Test
    Return split data and sample index masks """
    if X is None:
        X = data['X_raw']
    y = data['y_true']
    ds = data['datasets']

    train_mask = ds == 'Train'
    val_mask = ds == 'Validation'
    test_mask = ds == 'Test'

    return {
        'X_train': X[train_mask],
        'y_train': y[train_mask],
        'X_val': X[val_mask],
        'y_val': y[val_mask],
        'X_test': X[test_mask],
        'y_test': y[test_mask],
        'train_idx': np.where(train_mask)[0],
        'val_idx': np.where(val_mask)[0],
        'test_idx': np.where(test_mask)[0]
    }


def evaluate(y_true, y_pred, y_prob):
    """ Calculate binary classification metrics: Acc, Precision, Recall, F1, AUC """
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = 0.5

    return {
        'Accuracy': round(acc, 6),
        'Precision': round(prec, 6),
        'Recall': round(rec, 6),
        'F1': round(f1, 6),
        'AUC': round(auc, 6)
    }


def find_optimal_threshold(y_true, prob):
    """ Search threshold by maximize F1 score, using ROC curve thresholds """
    fpr, tpr, thresholds = roc_curve(y_true, prob)
    best_f1 = 0
    best_thresh = 0.5
    for thresh in thresholds:
        if thresh <= 0 or thresh >= 1:
            continue
        pred = (prob >= thresh).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh
    return best_thresh, best_f1


def save_results_table(results, category_name, output_dir):
    """ Save atomic performance table for all ensemble algorithms
    Sort by validation F1 descending """
    rows = []
    for algo_name, res in results.items():
        row = {'Algorithm': algo_name, 'Category': category_name}
        for split in ['train', 'val', 'test']:
            metrics = res[f'{split}_metrics']
            for k, v in metrics.items():
                row[f'{split}_{k}'] = v
        rows.append(row)

    df = pd.DataFrame(rows)
    df = df.sort_values('val_F1', ascending=False).reset_index(drop=True)
    path = os.path.join(output_dir, f'{category_name}_results.csv')
    df.to_csv(path, index=False)
    return df


def plot_roc_curves(results, category_name, output_dir, top_n=10):
    """ Plot ROC curves based on validation set """
    fig, ax = plt.subplots(figsize=(16, 12))
    sorted_res = sorted(results.items(), key=lambda x: x[1]['val_metrics']['F1'], reverse=True)
    cmap = plt.cm.tab20

    for i, (name, res) in enumerate(sorted_res[:top_n]):
        fpr, tpr, _ = roc_curve(res['y_val'], res['val_prob'])
        auc = res['val_metrics']['AUC']
        ax.plot(fpr, tpr, color=cmap(i % 20), linewidth=2.5, label=f'{name} (AUC={auc:.4f})')

    ax.plot([0, 1], [0, 1], 'k--', linewidth=2, alpha=0.5)
    ax.set_xlabel('False Positive Rate', fontsize=40)
    ax.set_ylabel('True Positive Rate', fontsize=40)
    ax.set_title(f'{category_name} - ROC Curves (Validation Set)', fontsize=44)
    ax.tick_params(axis='both', labelsize=36)
    ax.legend(fontsize=20, loc='lower right', framealpha=0.9)
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    plt.tight_layout()

    path = os.path.join(output_dir, f'{category_name}_ROC_curves.pdf')
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.close()


def plot_metric_comparison(results, category_name, output_dir):
    """ Bar chart comparison between validation and test set metrics """
    sorted_res = sorted(results.items(), key=lambda x: x[1]['val_metrics']['F1'], reverse=True)
    names = [r[0] for r in sorted_res]
    metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']

    fig, axes = plt.subplots(1, 5, figsize=(40, 12))
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0']

    for idx, metric in enumerate(metrics_list):
        ax = axes[idx]
        val_vals = [r[1]['val_metrics'][metric] for r in sorted_res]
        test_vals = [r[1]['test_metrics'][metric] for r in sorted_res]

        x = np.arange(len(names))
        width = 0.35
        ax.bar(x - width/2, val_vals, width, label='Validation', color=colors[idx], alpha=0.6)
        ax.bar(x + width/2, test_vals, width, label='Test', color=colors[idx], alpha=1.0)

        ax.set_ylabel(metric, fontsize=36)
        ax.set_title(metric, fontsize=40)
        ax.set_xticks(x)
        ax.set_xticklabels(names, rotation=45, ha='right', fontsize=18)
        ax.tick_params(axis='y', labelsize=32)
        ax.legend(fontsize=24)
        ax.set_ylim([0.5, 1.02])

    plt.suptitle(f'{category_name} - Metric Comparison', fontsize=48, y=1.02)
    plt.tight_layout()

    path = os.path.join(output_dir, f'{category_name}_metric_comparison.pdf')
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.close()


def save_sample_predictions(algo_name, data, pred_all, prob_all, raw_score_all, output_dir):
    """ Save per‑sample prediction for single algorithm
    Columns: Filename, True_Label, Predicted_Label, Raw_Score, Probability, Correct, Dataset """
    df = pd.DataFrame({
        'Filename': data['filenames'],
        'True_Label': data['y_true'],
        'Predicted_Label': pred_all.astype(int),
        'Raw_Score': raw_score_all,
        'Probability': prob_all,
        'Correct': pred_all.astype(int) == data['y_true'],
        'Dataset': data['datasets']
    })
    path = os.path.join(output_dir, f'{algo_name}_predictions.csv')
    df.to_csv(path, index=False)
    return df


# -------------------------- Voting Algorithm Definitions --------------------------
def absolute_majority_voting(params):
    n_needed = N_MODELS // 2 + 1
    votes_train = np.sum(X_train_pred, axis=1)
    votes_val = np.sum(X_val_pred, axis=1)
    votes_test = np.sum(X_test_pred, axis=1)
    prob_train = votes_train / N_MODELS
    prob_val = votes_val / N_MODELS
    prob_test = votes_test / N_MODELS
    return {'prob_train': prob_train, 'prob_val': prob_val, 'prob_test': prob_test}


def relative_majority_voting(params):
    pred_train = mode(X_train_pred, axis=1, keepdims=False).mode
    pred_val = mode(X_val_pred, axis=1, keepdims=False).mode
    pred_test = mode(X_test_pred, axis=1, keepdims=False).mode
    votes_train = np.sum(X_train_pred, axis=1)
    votes_val = np.sum(X_val_pred, axis=1)
    votes_test = np.sum(X_test_pred, axis=1)
    prob_train = votes_train / N_MODELS
    prob_val = votes_val / N_MODELS
    prob_test = votes_test / N_MODELS
    return {'prob_train': prob_train, 'prob_val': prob_val, 'prob_test': prob_test}


def weighted_voting(params):
    weights = np.array(params['weights'])
    weights = weights / weights.sum()
    votes_train = X_train_pred @ weights
    votes_val = X_val_pred @ weights
    votes_test = X_test_pred @ weights
    return {'prob_train': votes_train, 'prob_val': votes_val, 'prob_test': votes_test}


def hard_voting(params):
    pred_train = mode(X_train_pred, axis=1, keepdims=False).mode
    pred_val = mode(X_val_pred, axis=1, keepdims=False).mode
    pred_test = mode(X_test_pred, axis=1, keepdims=False).mode
    votes_train = np.sum(X_train_pred, axis=1) / N_MODELS
    votes_val = np.sum(X_val_pred, axis=1) / N_MODELS
    votes_test = np.sum(X_test_pred, axis=1) / N_MODELS
    return {'prob_train': votes_train, 'prob_val': votes_val, 'prob_test': votes_test}


def soft_voting(params):
    prob_train = np.mean(X_train_prob, axis=1)
    prob_val = np.mean(X_val_prob, axis=1)
    prob_test = np.mean(X_test_prob, axis=1)
    return {'prob_train': prob_train, 'prob_val': prob_val, 'prob_test': prob_test}


def probability_weighted_voting(params):
    weights = np.array(params['weights'])
    weights = weights / weights.sum()
    prob_train = X_train_prob @ weights
    prob_val = X_val_prob @ weights
    prob_test = X_test_prob @ weights
    return {'prob_train': prob_train, 'prob_val': prob_val, 'prob_test': prob_test}


def rank_voting(params):
    def rank_aggregate(prob_matrix):
        ranks = np.argsort(np.argsort(-prob_matrix, axis=1), axis=1)
        rank_scores = N_MODELS - ranks
        aggregated = np.mean(rank_scores, axis=1)
        aggregated = (aggregated - 1) / (N_MODELS - 1)
        return aggregated
    prob_train = rank_aggregate(X_train_prob)
    prob_val = rank_aggregate(X_val_prob)
    prob_test = rank_aggregate(X_test_prob)
    return {'prob_train': prob_train, 'prob_val': prob_val, 'prob_test': prob_test}


def bayesian_voting(params):
    prior = np.mean(y_train)
    alpha = params['alpha']
    def bayes_aggregate(prob_matrix):
        log_likelihood = np.log(np.clip(prob_matrix, 1e-10, 1-1e-10))
        log_prior = np.log(prior + 1e-10)
        log_posterior = np.mean(log_likelihood, axis=1) + alpha * log_prior
        log_likelihood_neg = np.log(np.clip(1 - prob_matrix, 1e-10, 1-1e-10))
        log_posterior_neg = np.mean(log_likelihood_neg, axis=1) + alpha * np.log(1 - prior + 1e-10)
        prob = np.exp(log_posterior) / (np.exp(log_posterior) + np.exp(log_posterior_neg) + 1e-10)
        return prob
    prob_train = bayes_aggregate(X_train_prob)
    prob_val = bayes_aggregate(X_val_prob)
    prob_test = bayes_aggregate(X_test_prob)
    return {'prob_train': prob_train, 'prob_val': prob_val, 'prob_test': prob_test}


def adaptive_voting(params):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    model_weights = np.zeros(N_MODELS)
    for i in range(N_MODELS):
        f1_scores = []
        for train_idx, val_idx in skf.split(X_train_prob, y_train):
            pred = (X_train_prob[val_idx, i] >= 0.5).astype(int)
            f1_scores.append(f1_score(y_train[val_idx], pred, zero_division=0))
        model_weights[i] = np.mean(f1_scores)
    model_weights = np.clip(model_weights, 1e-6, None)
    model_weights = model_weights / model_weights.sum()
    prob_train = X_train_prob @ model_weights
    prob_val = X_val_prob @ model_weights
    prob_test = X_test_prob @ model_weights
    return {'prob_train': prob_train, 'prob_val': prob_val, 'prob_test': prob_test}


def consensus_voting(params):
    threshold = params['threshold']
    def consensus_aggregate(pred_matrix, prob_matrix):
        agreement = np.std(pred_matrix, axis=1)
        consensus_mask = agreement <= threshold
        prob = np.mean(prob_matrix, axis=1)
        high_conf = np.max(prob_matrix, axis=1)
        low_conf = np.min(prob_matrix, axis=1)
        prob[~consensus_mask] = 0.5 * (high_conf[~consensus_mask] + low_conf[~consensus_mask])
        return prob
    prob_train = consensus_aggregate(X_train_pred, X_train_prob)
    prob_val = consensus_aggregate(X_val_pred, X_val_prob)
    prob_test = consensus_aggregate(X_test_pred, X_test_prob)
    return {'prob_train': prob_train, 'prob_val': prob_val, 'prob_test': prob_test}


# -------------------------- Main Experiment Pipeline --------------------------
CATEGORY = 'Voting'
OUT_DIR = os.path.join(OUTPUT_DIR, '04_Voting')
os.makedirs(OUT_DIR, exist_ok=True)

data = load_all_predictions()
splits_raw = split_by_dataset(data)
splits_prob = split_by_dataset(data, data['X_prob'])
splits_pred = split_by_dataset(data, data['X_pred'])

X_train_raw, y_train = splits_raw['X_train'], splits_raw['y_train']
X_val_raw, y_val = splits_raw['X_val'], splits_raw['y_val']
X_test_raw, y_test = splits_raw['X_test'], splits_raw['y_test']

X_train_prob = splits_prob['X_train']
X_val_prob = splits_prob['X_val']
X_test_prob = splits_prob['X_test']

X_train_pred = splits_pred['X_train']
X_val_pred = splits_pred['X_val']
X_test_pred = splits_pred['X_test']

train_idx = splits_raw['train_idx']
val_idx = splits_raw['val_idx']
test_idx = splits_raw['test_idx']

N_MODELS = len(MODELS)
results = {}


def run_voting_algo(name, fit_predict_fn, param_grid):
    """
    Hyper‑parameter search on TRAIN set, threshold optimize on TRAIN set
    Validation set for independent evaluation; Test set reserved for weight combination
    Save full sample predictions and store metrics into results dict
    """
    best_f1 = -1
    best_params = None
    best_state = None

    # Hyper‑parameter grid search using Train set
    for params in param_grid:
        state = fit_predict_fn(params)
        prob_eval = state['prob_train']
        thresh, _ = find_optimal_threshold(y_train, prob_eval)
        pred_eval = (prob_eval >= thresh).astype(int)
        f1 = f1_score(y_train, pred_eval, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_params = params
            best_state = state

    state = best_state
    prob_train = state['prob_train']
    prob_val = state['prob_val']
    prob_test = state['prob_test']

    # Optimize threshold on Train set only
    thresh, _ = find_optimal_threshold(y_train, prob_train)
    pred_train = (prob_train >= thresh).astype(int)
    pred_val = (prob_val >= thresh).astype(int)
    pred_test = (prob_test >= thresh).astype(int)

    # Assemble full‑length arrays for all samples
    n_samples = len(data['filenames'])
    pred_all = np.zeros(n_samples, dtype=int)
    prob_all = np.zeros(n_samples, dtype=float)
    raw_all = np.zeros(n_samples, dtype=float)

    pred_all[train_idx] = pred_train
    pred_all[val_idx] = pred_val
    pred_all[test_idx] = pred_test
    prob_all[train_idx] = prob_train
    prob_all[val_idx] = prob_val
    prob_all[test_idx] = prob_test
    raw_all[train_idx] = prob_train.copy()
    raw_all[val_idx] = prob_val.copy()
    raw_all[test_idx] = prob_test.copy()

    save_sample_predictions(name, data, pred_all, prob_all, raw_all, OUT_DIR)

    results[name] = {
        'best_params': best_params,
        'threshold': thresh,
        'train_metrics': evaluate(y_train, pred_train, prob_train),
        'val_metrics': evaluate(y_val, pred_val, prob_val),
        'test_metrics': evaluate(y_test, pred_test, prob_test),
        'y_val': y_val,
        'val_prob': prob_val,
        'y_test': y_test,
        'test_prob': prob_test
    }

    print(f'{name}: Val F1={results[name]["val_metrics"]["F1"]:.4f}, '
          f'Test F1={results[name]["test_metrics"]["F1"]:.4f}, '
          f'Val AUC={results[name]["val_metrics"]["AUC"]:.4f}')


# -------------------------- Launch Each Voting Algorithm --------------------------
run_voting_algo('Absolute_Majority_Voting', absolute_majority_voting, [{}])
run_voting_algo('Relative_Majority_Voting', relative_majority_voting, [{}])

# Weighted voting random weight grid
weight_grids = []
rng = np.random.RandomState(42)
for _ in range(50):
    w = rng.dirichlet(np.ones(N_MODELS))
    weight_grids.append({'weights': w.tolist()})
weight_grids.append({'weights': [1.0/N_MODELS] * N_MODELS})
run_voting_algo('Weighted_Voting', weighted_voting, weight_grids)

run_voting_algo('Hard_Voting', hard_voting, [{}])
run_voting_algo('Soft_Voting', soft_voting, [{}])

# Probability weighted voting random weight grid
prob_weight_grids = []
for _ in range(50):
    w = rng.dirichlet(np.ones(N_MODELS))
    prob_weight_grids.append({'weights': w.tolist()})
prob_weight_grids.append({'weights': [1.0/N_MODELS] * N_MODELS})
run_voting_algo('Probability_Weighted_Voting', probability_weighted_voting, prob_weight_grids)

run_voting_algo('Rank_Voting', rank_voting, [{}])
run_voting_algo(
    'Bayesian_Voting', bayesian_voting,
    [{'alpha': a} for a in [0.0, 0.1, 0.5, 1.0, 2.0]]
)
run_voting_algo('Adaptive_Voting', adaptive_voting, [{}])
run_voting_algo(
    'Consensus_Voting', consensus_voting,
    [{'threshold': t} for t in [0.1, 0.2, 0.3, 0.4, 0.5]]
)

# -------------------------- Export Basic Results --------------------------
df_results = save_results_table(results, CATEGORY, OUT_DIR)
plot_roc_curves(results, CATEGORY, OUT_DIR)
plot_metric_comparison(results, CATEGORY, OUT_DIR)

print(f'\n{CATEGORY} complete. Results saved to {OUT_DIR}')
print('\nValidation Set Performance (Independent Evaluation):')
print(df_results[['Algorithm', 'val_Accuracy', 'val_Precision', 'val_Recall', 'val_F1', 'val_AUC']].to_string(index=False))


Absolute_Majority_Voting: Val F1=0.7068, Test F1=0.8858, Val AUC=0.8224
Relative_Majority_Voting: Val F1=0.7068, Test F1=0.8858, Val AUC=0.8224
Weighted_Voting: Val F1=0.8131, Test F1=0.8850, Val AUC=0.8255
Hard_Voting: Val F1=0.7068, Test F1=0.8858, Val AUC=0.8224
Soft_Voting: Val F1=0.7247, Test F1=0.8918, Val AUC=0.8985
Probability_Weighted_Voting: Val F1=0.7756, Test F1=0.8891, Val AUC=0.9127
Rank_Voting: Val F1=0.6667, Test F1=0.8773, Val AUC=0.5000
Bayesian_Voting: Val F1=0.7441, Test F1=0.8917, Val AUC=0.9857
Adaptive_Voting: Val F1=0.7252, Test F1=0.8902, Val AUC=0.9082
Consensus_Voting: Val F1=0.7247, Test F1=0.8918, Val AUC=0.8985

Voting complete. Results saved to C:\Users\zhd\LungCancer\03.Ensemble learning\04_Voting

Validation Set Performance (Independent Evaluation):
                  Algorithm  val_Accuracy  val_Precision  val_Recall   val_F1  val_AUC
            Weighted_Voting        0.7715       0.687889       0.994 0.813088 0.825454
Probability_Weighted_Voting      

In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, precision_score, recall_score, f1_score
from sklearn.calibration import calibration_curve
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "axes.unicode_minus": False,
    "axes.titlesize": 40,
    "axes.labelsize": 40,
    "xtick.labelsize": 40,
    "ytick.labelsize": 40,
    "legend.fontsize": 40,
})

BASE_DIR = r'C:\Users\zhd\LungCancer'
OUTPUT_DIR = os.path.join(BASE_DIR, '03.Ensemble learning', '04_Voting')

COLOR_TRAIN = "#1f77b4"
COLOR_VAL = "#ff7f0e"
COLOR_TEST = "#2ca02c"


def safe_get(d, key):
    if key not in d:
        raise KeyError(f"Missing key '{key}'. Available keys: {list(d.keys())}")
    return d[key]


def debug_print_results_keys(results):
    print("\n==== DEBUG: keys in first algorithm result ====")
    first_key = list(results.keys())[0]
    print(f"Algorithm: {first_key}")
    print(f"Keys: {list(results[first_key].keys())}\n")


def bootstrap_metric_ci(y_true, y_pred, y_prob, n_boot=1000):
    rng = np.random.RandomState(42)
    metrics_boot = {"Accuracy": [], "Precision": [], "Recall": [], "F1": [], "AUC": []}
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        yt = y_true[idx]
        yp = y_pred[idx]
        ypr = y_prob[idx]
        try:
            acc = accuracy_score(yt, yp)
            prec = precision_score(yt, yp, zero_division=0)
            rec = recall_score(yt, yp, zero_division=0)
            f1 = f1_score(yt, yp, zero_division=0)
            auc = roc_auc_score(yt, ypr)
        except ValueError:
            continue
        metrics_boot["Accuracy"].append(acc)
        metrics_boot["Precision"].append(prec)
        metrics_boot["Recall"].append(rec)
        metrics_boot["F1"].append(f1)
        metrics_boot["AUC"].append(auc)

    out = {}
    for k, v in metrics_boot.items():
        arr = np.array(v)
        out[f"{k}_mean"] = np.mean(arr)
        out[f"{k}_ci_low"] = np.percentile(arr, 2.5)
        out[f"{k}_ci_high"] = np.percentile(arr, 97.5)
    return out


def bootstrap_auc_ci(y_true, y_score, n_boot=1000):
    rng = np.random.RandomState(42)
    aucs = []
    for _ in range(n_boot):
        idx = rng.choice(len(y_true), size=len(y_true), replace=True)
        try:
            auc = roc_auc_score(y_true[idx], y_score[idx])
            aucs.append(auc)
        except ValueError:
            continue
    aucs = np.array(aucs)
    mean_auc = roc_auc_score(y_true, y_score)
    ci_low = np.percentile(aucs, 2.5)
    ci_high = np.percentile(aucs, 97.5)
    return mean_auc, ci_low, ci_high


def save_fig(fig, base_name, out_dir, dpi=300):
    fig.savefig(os.path.join(out_dir, f"{base_name}.pdf"), dpi=dpi, bbox_inches="tight")
    fig.savefig(os.path.join(out_dir, f"{base_name}.png"), dpi=dpi, bbox_inches="tight")
    plt.close(fig)


def plot_metric_comparison(results, category_name, output_dir):
    sorted_res = sorted(results.items(), key=lambda x: safe_get(x[1]['val_metrics'], 'F1'), reverse=True)
    names = [r[0] for r in sorted_res]
    metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']

    fig, axes = plt.subplots(1, 5, figsize=(54, 20))
    colors = [COLOR_TRAIN, COLOR_TEST, COLOR_VAL]
    group_names = ["Train", "Test", "Validation"]
    width = 0.25
    x = np.arange(len(names))

    for idx_m, metric in enumerate(metrics_list):
        ax = axes[idx_m]
        train_vals = [safe_get(r[1]['train_metrics'], metric) for r in sorted_res]
        test_vals = [safe_get(r[1]['test_metrics'], metric) for r in sorted_res]
        val_vals = [safe_get(r[1]['val_metrics'], metric) for r in sorted_res]

        ax.bar(x - width, train_vals, width, label=group_names[0], color=colors[0], alpha=0.75)
        ax.bar(x, test_vals, width, label=group_names[1], color=colors[1], alpha=0.75)
        ax.bar(x + width, val_vals, width, label=group_names[2], color=colors[2], alpha=0.75)

        ax.set_ylabel(metric, fontsize=40)
        ax.set_title(metric, fontsize=40)
        ax.set_xticks(x)
        ax.set_xticklabels(names, rotation=40, ha="right", fontsize=40)
        ax.tick_params(axis='y', labelsize=40)
        ax.set_ylim([0.5, 1.02])

    fig.legend(labels=group_names, loc="lower center", ncol=3, fontsize=40, bbox_to_anchor=(0.5, 0.01))
    plt.suptitle(f"{category_name} Metric Comparison", fontsize=40, y=0.98)
    plt.tight_layout(rect=[0, 0.10, 1, 0.96])
    save_fig(fig, f"{category_name}_metric_comparison", output_dir)


def plot_roc_curves(results, category_name, output_dir):
    algo_list = list(results.keys())
    n_alg = len(algo_list)
    n_col = 3
    n_row = int(np.ceil(n_alg / n_col))
    sub_w, sub_h = 12, 12

    fig, axes = plt.subplots(n_row, n_col, figsize=(n_col * sub_w, n_row * sub_h))
    axes = axes.flatten()

    dataset_info = [
        ("Train", COLOR_TRAIN),
        ("Test", COLOR_TEST),
        ("Validation", COLOR_VAL),
    ]

    for idx, algo in enumerate(algo_list):
        ax = axes[idx]
        res = results[algo]
        for ds_name, color in dataset_info:
            if ds_name == "Train":
                train_m = res["train_metrics"]
                auc_val = train_m["AUC"]
                ax.plot([], [], color=color, lw=3, label=f"{ds_name} AUC={auc_val:.3f} (No raw data)")
            elif ds_name == "Test":
                y = safe_get(res, "y_test")
                prob = safe_get(res, "test_prob")
                mean_auc, ci_low, ci_high = bootstrap_auc_ci(y, prob)
                fpr, tpr, _ = roc_curve(y, prob)
                label_str = f"{ds_name} AUC={mean_auc:.3f} (95%CI:{ci_low:.3f}-{ci_high:.3f})"
                ax.plot(fpr, tpr, color=color, lw=3, label=label_str)
            elif ds_name == "Validation":
                y = safe_get(res, "y_val")
                prob = safe_get(res, "val_prob")
                mean_auc, ci_low, ci_high = bootstrap_auc_ci(y, prob)
                fpr, tpr, _ = roc_curve(y, prob)
                label_str = f"{ds_name} AUC={mean_auc:.3f} (95%CI:{ci_low:.3f}-{ci_high:.3f})"
                ax.plot(fpr, tpr, color=color, lw=3, label=label_str)

        ax.plot([0, 1], [0, 1], "k--", lw=3, alpha=0.6)
        ax.set_aspect("equal")
        ax.set_xlabel("False Positive Rate", fontsize=40)
        ax.set_ylabel("True Positive Rate", fontsize=40)
        ax.set_title(f"{algo}", fontsize=40)
        ax.tick_params(axis="both", labelsize=40)
        ax.legend(loc="lower right", frameon=True, fontsize=40)
        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.02, 1.02)

    for idx in range(len(algo_list), len(axes)):
        axes[idx].axis("off")

    plt.suptitle(f"{category_name} ROC Curves (Train / Test / Validation)", fontsize=40, y=1.03)
    plt.tight_layout()
    save_fig(fig, f"{category_name}_ROC_curves", output_dir)


def plot_calibration_all(results, category_name, output_dir):
    algo_list = list(results.keys())
    n_alg = len(algo_list)
    n_col = 3
    n_row = int(np.ceil(n_alg / n_col))
    sub_w, sub_h = 16, 12

    fig, axes = plt.subplots(n_row, n_col, figsize=(n_col * sub_w, n_row * sub_h))
    axes = axes.flatten()

    ds_spec = [
        ("Train", COLOR_TRAIN),
        ("Test", COLOR_TEST),
        ("Val", COLOR_VAL),
    ]

    for i, algo in enumerate(algo_list):
        ax = axes[i]
        res = results[algo]
        for ds_name, c in ds_spec:
            if ds_name == "Train":
                ax.plot([], [], color=c, lw=3, marker="o", label=f"{ds_name}: No raw probability")
            elif ds_name == "Test":
                y_true = safe_get(res, "y_test")
                prob = safe_get(res, "test_prob")
                prob_clipped = np.clip(prob, 0.0, 1.0)
                frac_pos, mean_pred = calibration_curve(y_true, prob_clipped, n_bins=10)
                ax.plot(mean_pred, frac_pos, marker="o", lw=3, label=ds_name, color=c)
            elif ds_name == "Val":
                y_true = safe_get(res, "y_val")
                prob = safe_get(res, "val_prob")
                prob_clipped = np.clip(prob, 0.0, 1.0)
                frac_pos, mean_pred = calibration_curve(y_true, prob_clipped, n_bins=10)
                ax.plot(mean_pred, frac_pos, marker="o", lw=3, label=ds_name, color=c)

        ax.plot([0, 1], [0, 1], "k--", lw=3)
        ax.set_xlabel("Mean predicted probability", fontsize=40)
        ax.set_ylabel("Fraction of positives", fontsize=40)
        ax.set_title(f"{algo} Calibration Curve", fontsize=40)
        ax.legend(fontsize=40)
        ax.tick_params(labelsize=40)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)

    for j in range(len(algo_list), len(axes)):
        axes[j].axis("off")

    plt.suptitle(f"{category_name} Calibration Curves (Train / Test / Validation)", fontsize=40, y=1.03)
    plt.tight_layout()
    save_fig(fig, f"{category_name}_calibration", output_dir)


def plot_dca_all(results, category_name, output_dir):
    algo_list = list(results.keys())
    n_alg = len(algo_list)
    n_col = 3
    n_row = int(np.ceil(n_alg / n_col))
    sub_w, sub_h = 12, 12

    fig, axes = plt.subplots(n_row, n_col, figsize=(n_col * sub_w, n_row * sub_h))
    axes = axes.flatten()
    threshold_range = np.linspace(0.01, 0.99, 100)

    ds_spec = [
        ("Train", COLOR_TRAIN),
        ("Test", COLOR_TEST),
        ("Val", COLOR_VAL),
    ]

    for idx, algo in enumerate(algo_list):
        ax = axes[idx]
        res = results[algo]
        for ds_name, color in ds_spec:
            if ds_name == "Train":
                ax.plot([], [], color=color, lw=3, label=f"{ds_name}: No raw data")
            elif ds_name == "Test":
                y = safe_get(res, "y_test")
                prob = safe_get(res, "test_prob")
                net_benefit = []
                for t in threshold_range:
                    pred = (prob >= t).astype(int)
                    tp = np.sum((pred == 1) & (y == 1))
                    fp = np.sum((pred == 1) & (y == 0))
                    n = len(y)
                    nb = (tp / n) - (fp / n) * (t / (1 - t)) if not (t == 0 or t == 1) else 0
                    net_benefit.append(nb)
                ax.plot(threshold_range, net_benefit, lw=3, label=ds_name, color=color)
            elif ds_name == "Val":
                y = safe_get(res, "y_val")
                prob = safe_get(res, "val_prob")
                net_benefit = []
                for t in threshold_range:
                    pred = (prob >= t).astype(int)
                    tp = np.sum((pred == 1) & (y == 1))
                    fp = np.sum((pred == 1) & (y == 0))
                    n = len(y)
                    nb = (tp / n) - (fp / n) * (t / (1 - t)) if not (t == 0 or t == 1) else 0
                    net_benefit.append(nb)
                ax.plot(threshold_range, net_benefit, lw=3, label=ds_name, color=color)

        ax.axhline(y=0, color="#777777", linestyle="--", label="Treat none", lw=3)
        ax.set_xlabel("Threshold probability", fontsize=40)
        ax.set_ylabel("Net benefit", fontsize=40)
        ax.set_title(f"{algo} DCA", fontsize=40)
        ax.legend(fontsize=40)
        ax.tick_params(labelsize=40)

    for j in range(len(algo_list), len(axes)):
        axes[j].axis("off")

    plt.suptitle(f"{category_name} Decision Curve Analysis (DCA)", fontsize=40, y=1.03)
    plt.tight_layout()
    save_fig(fig, f"{category_name}_DCA", output_dir)


def plot_nomogram_density_all(results, category_name, output_dir):
    algo_list = list(results.keys())
    n_alg = len(algo_list)
    n_col = 3
    n_row = int(np.ceil(n_alg / n_col))
    sub_w, sub_h = 12, 12

    fig = plt.figure(figsize=(n_col * sub_w, n_row * sub_h))
    gs = GridSpec(n_row, n_col, figure=fig)

    for idx, algo in enumerate(algo_list):
        row_pos = idx // n_col
        col_pos = idx % n_col
        res = results[algo]
        ax_main = fig.add_subplot(gs[row_pos, col_pos])

        prob_all = np.concatenate([safe_get(res, "test_prob"), safe_get(res, "val_prob")])
        y_all = np.concatenate([safe_get(res, "y_test"), safe_get(res, "y_val")])

        ax_main.hist(prob_all, bins=30, density=True, alpha=0.25, color="#aaaaaa", zorder=0)
        jitter = np.random.normal(0, 0.04, size=len(prob_all))

        for label_val, c, marker in zip([0, 1], ["#1f77b4", "#d62728"], ["o", "^"]):
            mask = y_all == label_val
            ax_main.scatter(prob_all[mask], label_val + jitter[mask], s=60, alpha=0.65, color=c, marker=marker)

        ax_main.set_xlim(0, 1)
        ax_main.set_ylim(-0.3, 1.3)
        ax_main.set_yticks([0, 1])
        ax_main.set_yticklabels(["Negative", "Positive"], fontsize=40)
        ax_main.set_xlabel("Predicted Probability", fontsize=40)
        ax_main.set_title(f"{algo} Nomogram‑Density", fontsize=40)
        ax_main.tick_params(labelsize=40)

    plt.suptitle(f"{category_name} Nomogram with Sample Density", fontsize=40, y=1.03)
    plt.tight_layout()
    save_fig(fig, f"{category_name}_nomogram_density", output_dir)


def merge_all_predictions(results, output_dir):
    collect = []
    for algo_name in results.keys():
        csv_path = os.path.join(output_dir, f"{algo_name}_predictions.csv")
        df = pd.read_csv(csv_path)
        df = df.copy()
        df.loc[df["Raw_Score"].isna(), "Raw_Score"] = "--"
        df = df.rename(columns={
            "Predicted_Label": f"{algo_name}_PredLabel",
            "Raw_Score": f"{algo_name}_RawScore",
            "Probability": f"{algo_name}_Prob",
        })
        keep_cols = ["Filename", "True_Label", "Dataset",
                     f"{algo_name}_PredLabel", f"{algo_name}_RawScore", f"{algo_name}_Prob"]
        collect.append(df[keep_cols])

    base_df = collect[0][["Filename", "True_Label", "Dataset"]].copy()
    for df_sub in collect:
        merge_cols = ["Filename", "True_Label", "Dataset"]
        base_df = pd.merge(base_df, df_sub, on=merge_cols, how="left")

    out_file = os.path.join(output_dir, "all_predictions.csv")
    base_df.to_csv(out_file, index=False)
    print(f"Merged prediction saved: {out_file}")
    return base_df


def export_all_evaluation_with_ci(results, output_dir, filename="all_evaluation_metrics.csv"):
    metric_list = ["Accuracy", "Precision", "Recall", "F1", "AUC"]
    rows = []

    for algo_name, res in results.items():
        for ds_name in ["Train", "Val", "Test"]:
            row = {
                "Category": "Voting",
                "Algorithm": algo_name,
                "Dataset": ds_name
            }
            if ds_name == "Train":
                m = res["train_metrics"]
                for met in metric_list:
                    row[met] = m[met]
                    row[f"{met}_CI95_low"] = np.nan
                    row[f"{met}_CI95_high"] = np.nan
            elif ds_name == "Val":
                y_true = safe_get(res, "y_val")
                thresh = safe_get(res, "threshold")
                y_pred = (safe_get(res, "val_prob") >= thresh).astype(int)
                y_prob = safe_get(res, "val_prob")
                ci_dict = bootstrap_metric_ci(y_true, y_pred, y_prob, n_boot=1000)
                for met in metric_list:
                    row[met] = round(ci_dict[f"{met}_mean"], 6)
                    row[f"{met}_CI95_low"] = round(ci_dict[f"{met}_ci_low"], 6)
                    row[f"{met}_CI95_high"] = round(ci_dict[f"{met}_ci_high"], 6)
            elif ds_name == "Test":
                y_true = safe_get(res, "y_test")
                thresh = safe_get(res, "threshold")
                y_pred = (safe_get(res, "test_prob") >= thresh).astype(int)
                y_prob = safe_get(res, "test_prob")
                ci_dict = bootstrap_metric_ci(y_true, y_pred, y_prob, n_boot=1000)
                for met in metric_list:
                    row[met] = round(ci_dict[f"{met}_mean"], 6)
                    row[f"{met}_CI95_low"] = round(ci_dict[f"{met}_ci_low"], 6)
                    row[f"{met}_CI95_high"] = round(ci_dict[f"{met}_ci_high"], 6)
            rows.append(row)

    df_out = pd.DataFrame(rows)
    csv_path = os.path.join(output_dir, filename)
    df_out.to_csv(csv_path, index=False)
    print(f"Evaluation table with bootstrap CI saved: {csv_path}")
    return df_out


# ========== Execute ========== 
CATEGORY = "Voting"
os.makedirs(OUTPUT_DIR, exist_ok=True)

debug_print_results_keys(results)
plot_metric_comparison(results, CATEGORY, OUTPUT_DIR)
plot_roc_curves(results, CATEGORY, OUTPUT_DIR)
plot_calibration_all(results, CATEGORY, OUTPUT_DIR)
plot_dca_all(results, CATEGORY, OUTPUT_DIR)
plot_nomogram_density_all(results, CATEGORY, OUTPUT_DIR)
merge_all_predictions(results, OUTPUT_DIR)
export_all_evaluation_with_ci(results, OUTPUT_DIR, filename="all_evaluation_metrics.csv")

print(f"All figures & tables saved to {OUTPUT_DIR}")



==== DEBUG: keys in first algorithm result ====
Algorithm: Absolute_Majority_Voting
Keys: ['best_params', 'threshold', 'train_metrics', 'val_metrics', 'test_metrics', 'y_val', 'val_prob', 'y_test', 'test_prob']

Merged prediction saved: C:\Users\zhd\LungCancer\03.Ensemble learning\04_Voting\all_predictions.csv
Evaluation table with bootstrap CI saved: C:\Users\zhd\LungCancer\03.Ensemble learning\04_Voting\all_evaluation_metrics.csv
All figures & tables saved to C:\Users\zhd\LungCancer\03.Ensemble learning\04_Voting
